## 주가 데이터 분석

- `yfinance` 라이브러리를 사용하여 주가 데이터를 수집합니다. (2020-01-01 ~ 현재)
- 날짜 인덱스와 결측 개념을 확인합니다.
- 일간 수익률과 누적 수익률을 계산합니다.
- 장기 투자 성과를 선 그래프로 시각화합니다.

### 필요한 모듈 임포트

In [ ]:
# Colab에서 필요한 라이브러리 설치 여부를 확인합니다.
# [참고] 코드 실행 결과 콘솔에 아무것도 출력되지 않으면 현재 Colab에는
# 'yfinance'를 포함하는 라이브러리가 설치되지 않았다는 것을 의미합니다.
!pip list | grep yfinance

In [ ]:
# 필요한 라이브러리를 설치합니다.
# !pip install yfinance

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import yfinance as yf

### 주가 데이터 수집

In [ ]:
# 티커를 설정합니다.
ticker = '005930.KS'

In [ ]:
# 주가 데이터 수집 시작일자를 설정합니다.
# [참고] 단순 수익률을 계산할 때 첫 번째 행은 결측이 되므로 시작일자를
# 데이터 수집 기간의 시작일보다 하루 앞당긴 날자로 지정합니다.
start_date = '2019-12-31'

In [ ]:
# 주가 데이터 수집 종료일자를 설정합니다.
# [참고] 주가 수집 함수가 마지막 일자를 포함하지 않습니다.
# 아래 코드를 실행하면 시스템 날짜를 end_date에 할당합니다.
end_date = pd.Timestamp.today().strftime(format='%Y-%m-%d')

In [ ]:
# 주가 데이터를 수집하고 데이터프레임을 생성합니다.
# [참고] end 매개변수에 지정한 날짜를 포함하지 않습니다.
# auto_adjust 매개변수를 생략하면 배당과 액면분할 반영하여 OLHC를 조정합니다.
df = yf.download(tickers=ticker, start=start_date, end=end_date)

### 데이터프레임 구조 확인

In [ ]:
# df의 처음 5행을 확인합니다.
# [참고] n 매개변수에 출력할 행 개수를 정수로 설정합니다.(기본값: 5)
df.head()

In [ ]:
# df의 열이름을 확인합니다.
# [참고] 열이름이 멀티 인덱스로 생성되었습니다.
# 열이름 네임은 ['Price', 'Ticker']로 설정되었습니다.
df.columns

In [ ]:
# df의 열이름에서 첫 번째 레벨만 남깁니다.
# [참고] df.columns.droplevel(level=1)은 두 번째 레벨을 삭제합니다.
df.columns = df.columns.get_level_values(level=0)

In [ ]:
# df의 열이름을 확인합니다.
# [참고] df의 열이름 네임이 'Price'로 남아 있습니다.
df.columns

In [ ]:
# df의 열이름 네임을 삭제합니다.
# [참고] df.columns.name에 None을 할당하면 열이름 네임을 초기화합니다.
df.columns.name = None

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

In [ ]:
# 주가 데이터를 수집하고 데이터프레임을 생성합니다.
# [참고] auto_adjust 매개변수에 False를 지정하면 OHLC를 유지하고 수정 종가를 추가합니다.(기본값: True)
# progress 매개변수에 False를 지정하면 다운로드 진행 막대를 출력하지 않습니다.(기본값: True)
# multi_level_index 매개변수에 False를 지정하면 단일 레벨 열이름을 반환합니다.(기본값: True)
df = yf.download(
    tickers=ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
    multi_level_index=False,
)

### 데이터프레임 전처리

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

In [ ]:
# df의 열 순서를 변경하기 위해 열이름 리스트를 생성합니다.
# [참고] 이번 실습에서 수정 종가를 사용하지 않을 예정이므로 제외합니다.
cols = ['Open', 'High', 'Low', 'Close', 'Volume']

In [ ]:
# df의 열 순서를 변경하고 df에 재할당합니다.
df = df.loc[:, cols]

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

### 날짜 인덱스 전처리

In [ ]:
# df의 정보를 확인합니다.
df.info()

In [ ]:
# df의 인덱스를 확인합니다.
# [참고] df의 인덱스는 DatetimeIndex입니다.
df.index

In [ ]:
# df의 인덱스를 초기화하고 기존 인덱스를 첫 번째 열로 삽입한 결과를 df에 재할당합니다.
df = df.reset_index()

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

In [ ]:
# df에서 Date를 문자열로 변환합니다.
df['Date'] = df['Date'].astype(dtype=str)

In [ ]:
# df의 열별 자료형을 확인합니다.
df.dtypes

In [ ]:
# df의 행 순서를 무작위로 섞은 결과를 df에 재할당합니다.
df = df.sample(frac=1, random_state=1, ignore_index=True)

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

In [ ]:
# df에서 Date를 날짜시간형으로 변환합니다.
df['Date'] = pd.to_datetime(arg=df['Date'])

In [ ]:
# df에서 Date를 인덱스로 설정하고 df에 재할당합니다.
df = df.set_index(keys='Date')

In [ ]:
# df를 인덱스 기준으로 오름차순 정렬하고 df에 재할당합니다.
df = df.sort_index()

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

### [참고] 누락된 인덱스 추가 및 결측값 처리

- 주식 데이터는 거래일에만 값이 존재합니다.
- 따라서 누락된 인덱스는 주말 또는 공휴일입니다.
- 만약 누락된 인덱스를 추가하면 해당일은 결측값(`np.nan`)으로 채워집니다.
- 연속적인 시계열 데이터는 누락된 인덱스를 추가하고 선형 보간법 등을 사용하여 결측을 채웁니다.

In [ ]:
# df의 인덱스를 일(day) 단위로 재생성하고 처음 5행만 선택하여 df_na에 할당합니다.
df_na = df.asfreq(freq='D').head()

In [ ]:
# df_na를 확인합니다.
df_na

In [ ]:
# df_na에서 종가에 있는 결측값을 직전 값으로 채운 결과를 확인합니다.
df_na['Close'].ffill()

In [ ]:
# df_na에서 종가에 있는 결측값을 선형 보간법으로 채운 결과를 확인합니다.
# [주의] 주식 데이터의 결측값을 선형 보간법으로 함부로 채우면 안 됩니다.
df_na['Close'].interpolate(method='linear')

### [참고] 전역 변수 목록 확인 및 변수 삭제

In [ ]:
# 전역 변수 목록을 확인합니다.
%whos

In [ ]:
# 전역 변수 목록에서 데이터프레임만 확인합니다.
%whos DataFrame

In [ ]:
# df_na를 전역 변수 목록에서 삭제합니다.
del df_na

In [ ]:
# 전역 변수 목록에서 데이터프레임만 확인합니다.
%whos DataFrame

### 날짜 인덱스 필터링

In [ ]:
# df에서 2020년도 데이터만 선택합니다.(인덱싱)
# [참고] df에는 'yyyy' 형태의 행이름은 없지만, df의 인덱스가 날짜 인덱스이므로
# 네자리 정수를 지정하여 원하는 연도를 인덱싱할 수 있습니다.
df.loc['2020']

In [ ]:
# df에서 2020년 1월 데이터만 선택합니다.(인덱싱)
# [참고] 'yyyy-mm' 형태의 지정하여 특정 연도와 월까지 함께 지정할 수 있습니다.
# [주의] 인덱스의 부분 문자열에 대해 팬시 인덱싱과 슬라이싱을 지원하지 않습니다.
df.loc['2020-01']

In [ ]:
# df에서 2020년 1월 2일과 3일 데이터만 선택합니다.(팬시 인덱싱)
df.loc[['2020-01-02', '2020-01-03']]

In [ ]:
# df에서 2020년 1월 1~7일 데이터만 선택합니다.(슬라이싱)
df.loc['2020-01-01':'2020-01-07']

### 날짜 인덱스의 속성 활용

In [ ]:
# df에서 매년 1월 데이터만 선택합니다.(불리언 인덱싱)
df.loc[df.index.month == 1]

In [ ]:
# df에서 매년 12월 31일 데이터만 선택합니다.
df.loc[(df.index.month == 12) & (df.index.day == 31)]

In [ ]:
# df의 인덱스에서 정수 요일을 생성합니다.
# [참고] 정수 0은 Monday, 정수 6은 Sunday입니다.
df.index.dayofweek

In [ ]:
# df에서 매주 월요일 데이터만 선택합니다.
df.loc[df.index.dayofweek == 0]

In [ ]:
# df의 인덱스에서 영문 요일 문자열을 생성합니다.
# [참고] date_format 매개변수에 지정한 날짜 포맷에 맞는 문자열을 반환합니다.
df.index.strftime(date_format='%A')

In [ ]:
# df의 인덱스에서 영문 요일 문자열을 생성합니다.
# [참고] locale 매개변수에 'ko_KR'을 지정하면 요일을 한글 문자열로 반환합니다.
# [주의] Colab 환경에서는 한국 로케일을 바로 사용할 수 없습니다.
df.index.day_name()

In [ ]:
# 한글 요일 문자열을 매핑할 딕셔너리를 생성합니다.
weekday_kor = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}

In [ ]:
# df의 인덱스에서 정수 요일을 추출하고, 한글 요일명으로 매핑합니다.
df.index.dayofweek.map(mapper=weekday_kor)

### [참고] 집계 함수 활용법

In [ ]:
# df에서 매년 첫 번째 거래일 데이터만 선택합니다.
# [참고] first 메서드는 그룹 집계 함수이므로, 그룹 키를 새로운 인덱스로 설정합니다.
df.groupby(by=df.index.year).first()

In [ ]:
# df에서 매년 마지막 거래일 데이터만 선택합니다.
df.groupby(by=df.index.year).last()

In [ ]:
# df에서 매년 첫 번째 거래일 데이터만 선택합니다.
# [참고] nth 메서드는 n번째 행을 선택하는 함수이므로, 원본 인덱스를 유지합니다.
df.groupby(by=df.index.year).nth(n=0)

In [ ]:
# df에서 매년 마지막 거래일 데이터만 선택합니다.
df.groupby(by=df.index.year).nth(n=-1)

### 수익률 계산

- 이번 예제에서 다루는 수익률은 투자 수익률이 아니고 가격 변화율(시장 수익률)입니다.
- 투자 수익률은 체결(진입/청산) 가격을 기준으로 계산해야 합니다.

In [ ]:
# 단순 수익률을 계산하고 df에 추가합니다.
# [참고] pct_change 메서드는 직전 값 대비 현재 값의 변화율을 반환합니다.
# 따라서 실제 투자 수익률과는 다릅니다.
df['Daily_Return'] = df['Close'].pct_change()

In [ ]:
# 로그 수익률을 계산하고 df에 추가합니다.
# [참고] 로그 수익률은 시계열 분석 및 통계 모델링에 더 적합합니다.
# shift 메서드는 periods 매개변수에 지정한 정수만큼 위(음수), 아래(양수)로 이동시킵니다.
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(periods=1))

In [ ]:
# df에서 열별 결측값 개수를 확인합니다.
df.isna().sum()

In [ ]:
# df에서 결측값을 0으로 대체하고 df에 재할당합니다.
# [참고] dropna 메서드의 axis 매개변수에 0을 지정하면 결측을 포함하는 행,
# 1을 지정하면 결측을 포함하는 열을 삭제합니다.(기본값: 0)
df = df.fillna(value=0)

In [ ]:
# 누적 수익률을 계산하고 df에 추가합니다.
# [참고] 단순 수익률에 1을 더하여 원금을 1로 설정합니다.
# Cum_Return은 매 시점의 종가가 초기 대비 몇 배가 되었는지를 의미합니다.
df['Cum_Return'] = (df['Daily_Return'] + 1).cumprod()

In [ ]:
# df의 처음 10행을 확인합니다.
df.head(n=10)

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

In [ ]:
# 로그 누적 수익률을 계산하고 일반 수익률로 변환합니다.
# [참고] 단순 수익률을 누적하려면 곱셈이 필요하지만, 로그 수익률은 합산합니다.
# 로그 누적 수익률을 지수 변환하면 단순 누적 수익률과 동일한 값이 됩니다.
np.exp(df['Log_Return'].cumsum())

In [ ]:
# 누적 수익률의 백분율을 계산하고 df에 추가합니다.
# [참고] Cum_Return에서 1을 차감하고 100을 곱하여 순수 수익률을 확인합니다.
# 누적 수익률은 1을 기준으로 배수 형태로 계산하지만, 실제로 해석할 때에는
# 원금을 제외한 수익 부분이 중요하므로 누적 수익률에서 1을 차감합니다.
df['Cum_Return_Pct'] = (df['Cum_Return'] - 1) * 100

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

In [ ]:
# 시작 종가와 마지막 종가를 각각 생성합니다.
base_close, last_close = df['Close'].iloc[[0, -1]]

In [ ]:
# 전체 기간 누적 수익률을 확인합니다.
(last_close / base_close - 1) * 100

### 변동성 계산

In [ ]:
# 단순 수익률의 평균을 확인합니다.
df['Daily_Return'].mean()

In [ ]:
# 단순 수익률의 표준편차를 확인합니다.
df['Daily_Return'].std()

In [ ]:
# 단순 수익률의 20일 이동 표준편차를 계산하고 df에 추가합니다.
df['Volatility_20'] = df['Daily_Return'].rolling(window=20).std()

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

### 이동 평균 계산

In [ ]:
# 종가의 20일 이동 평균을 계산하고 df에 추가합니다.
df['Moving_Avg_20'] = df['Close'].rolling(window=20).mean()

In [ ]:
# 종가의 60일 이동 평균을 계산하고 df에 추가합니다.
df['Moving_Avg_60'] = df['Close'].rolling(window=60).mean()

In [ ]:
# df의 처음 10행을 확인합니다.
df.head(n=10)

In [ ]:
# df의 마지막 10행을 확인합니다.
df.tail(n=10)

### Colab에서 한글 폰트 설치

In [ ]:
# 현재 Colab에 설치된 폰트 목록을 확인합니다.
!fc-list

In [ ]:
# 불필요한 경로를 제거하고 전체 폰트 이름만 출력합니다.
!fc-list : family

In [ ]:
# Colab에서 사용할 나눔 폰트를 설치합니다.
!apt-get install -y fonts-nanum

In [ ]:
# 불필요한 경로를 제거하고 한글 폰트 이름만 중복 없이 오름차순 정렬하여 출력합니다.
!fc-list :lang=ko family | sort | uniq

### 한글 폰트명 탐색

In [ ]:
# 필요한 모듈을 임포트합니다.
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 현재 사용 중인 컴퓨터에 설치된 전체 폰트 파일명을 리스트로 생성합니다.
fontList = fm.findSystemFonts(fontext='ttf')

In [ ]:
# fontList의 원소 개수를 확인합니다.
len(fontList)

In [ ]:
# fontList에서 특정 폰트명을 포함하는 파일명을 선택하여 fontPath에 할당합니다.
fontPath = sorted([font for font in fontList if 'Nanum' in font])

In [ ]:
# fontPath를 확인합니다.
fontPath

In [ ]:
# 반복문으로 컴퓨터에 설치된 폰트명을 리스트로 반환합니다.
[fm.FontProperties(fname=font).get_name() for font in fontPath]

In [ ]:
# matplotlib 라이브러리의 폰트 관리자 객체를 초기화합니다.
# [참고] 컴퓨터에 설치된 폰트 파일들을 다시 스캔하여 내부 폰트 캐시를 재구성하여
# 새로 설치한 한글 폰트를 사용할 수 있게 합니다.
fm.fontManager.__init__()

### 그래픽 요소 설정

In [ ]:
# 한글 폰트, 그래프 크기와 해상도 등 그래픽 요소를 설정합니다.
plt.rc(group='font', family='NanumBarunGothic', size=10)
plt.rc(group='figure', figsize=(12, 4), dpi=120)
plt.rc(group='axes', unicode_minus=False)
plt.rc(group='legend', frameon=True, fc='0.9', ec='0.9')

### 장기 투자 성과 시각화

In [ ]:
# 종가 추이를 선 그래프로 시각화합니다.
# [참고] color 매개변수에 0~1 범위의 실수를 문자열로 지정할 수 있습니다.
# '0'은 'black', '1'은 'white'이고, '0.5'는 'gray'입니다.
sns.lineplot(data=df, x=df.index, y='Close', color='0', lw=1)
plt.axhline(y=base_close, color='0.5', ls='--', lw=0.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='가격($)')
plt.title(label=f'{ticker} 종가($)', fontweight='bold')
plt.show()

In [ ]:
# 누적 수익률 추이를 선 그래프로 시각화합니다.
sns.lineplot(data=df, x=df.index, y='Cum_Return_Pct', color='red', lw=1)
plt.axhline(y=0, color='0.5', ls='--', lw=0.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='백분율(%)')
plt.title(label=f'{ticker} 누적 수익률 백분율(%)', fontweight='bold')
plt.show()

In [ ]:
# 로그 수익률의 기술통계량을 확인합니다.
df['Log_Return'].describe()

In [ ]:
# 로그 수익률을 히스토그램으로 시각화합니다.
# [참고] fc 매개변수는 face color(채우기 색), ec 매개변수는 edge color(테두리 색)을 의미합니다.
# 로그 수익률은 정규분포처럼 보이지만 실제로는 양쪽 꼬리가 정규분포보다 두꺼운 특징을 가집니다.
sns.histplot(data=df, x='Log_Return', fc='0.8', ec='1',
             binrange=(-0.15, 0.15), bins=60, stat='density')
sns.kdeplot(data=df, x='Log_Return', color='red', lw=1.5)
plt.title(label=f'{ticker} 로그 수익률', fontweight='bold')
plt.show()

In [ ]:
# 로그 수익률의 분포를 상자 그림으로 시각화합니다.
sns.boxplot(data=df, y='Log_Return', color='0.8', width=0.3, fliersize=5,
            flierprops={'markerfacecolor': 'pink', 'markeredgecolor': 'red'})
plt.xlabel(xlabel=f'{ticker}')
plt.ylabel(ylabel='로그 수익률')
plt.title(label=f'{ticker} 로그 수익률 상자 그림', fontweight='bold')
plt.show()

In [ ]:
# 필요한 모듈을 임포트합니다.
from scipy import stats

In [ ]:
# 로그 수익률을 Normal Q-Q Plot으로 시각화합니다.
# [참고] 모든 점이 빨간색 직선 위에 놓여 있으면 데이터가 정규분포한다고 판단합니다.
stats.probplot(x=df['Log_Return'], dist='norm', plot=plt)
plt.xlabel(xlabel='정규분포 기준 이론적 분위수')
plt.ylabel(ylabel='표본 데이터 분위수')
plt.title(label=f'{ticker} Normal Q-Q Plot', fontweight='bold')
plt.show()

In [ ]:
# 이동 변동성을 선 그래프로 시각화합니다.
sns.lineplot(data=df, x=df.index, y='Volatility_20', color='red', lw=1)
plt.axhline(y=df['Volatility_20'].mean(), color='0.5', ls='--', lw=0.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='표준편차')
plt.title(label=f'{ticker} 20일 이동 표준편차', fontweight='bold')
plt.show()

In [ ]:
# 이동 평균을 선 그래프로 시각화합니다.
sns.lineplot(data=df, x=df.index, y='Close', color='0.8', lw=1)
sns.lineplot(data=df, x=df.index, y='Moving_Avg_20', color='blue', lw=1)
sns.lineplot(data=df, x=df.index, y='Moving_Avg_60', color='red', lw=1.5)
plt.xlabel(xlabel='거래일')
plt.ylabel(ylabel='가격($)')
plt.title(label=f'{ticker} 종가 및 이동 평균', fontweight='bold')
plt.show()

## End of Document